# Module 30 — The Pretraining Corpus

The README's scope decision targets **~125M params on ~2.5B tokens** to
start (Chinchilla-optimal, fits Colab Pro's included compute budget). This
module builds the real data pipeline for that corpus — sourcing real
datasets, mixing them, tokenizing, and writing a training-ready binary file
— and verifies it end to end at a **small scale** (a couple hundred
thousand tokens) before Module 31 points the same pipeline at the full 2.5B
target.

**Sources** (both real, existing datasets — nothing fabricated):
- **General English text (real-world knowledge):**
  [`HuggingFaceFW/fineweb-edu`](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) —
  a modern, filtered web-text corpus specifically curated for
  *educational/informative* content, streamed (it's enormous — streaming
  avoids downloading more than what's actually needed).
- **Genshin specialization:**
  [`mrzjy/multimodal-genshin-impact`](https://huggingface.co/datasets/mrzjy/multimodal-genshin-impact) —
  the same 22,162-page Fandom wiki dump identified earlier in this
  project's research. Only ~47.6M tokens total — far smaller than the
  2.5B-token budget, so it gets **repeated** (looped) enough times to reach
  its target share, same as any domain-specialization slice in a larger
  pretraining mix.
- **Mixing:** target ~10% Genshin, ~90% general text — enough repetition
  for real specialization without the corpus lop-sidedly overweighting a
  47.6M-token source, informed by Module 18's lesson about a model
  memorizing a training set that's too small relative to its capacity.

## 1. Tokenizing the Genshin wiki text (repeated to hit its token share)

Same wiki dump, but now for **pretraining** — raw prose text, not the
instruction/QA pairs an earlier fine-tuning-focused exploration of this
project once built (that approach was abandoned in favor of training from
scratch; see the project's scope-history notes). Uses Module 20's real
`tiktoken` GPT-2 tokenizer, with GPT-2's `<|endoftext|>` token as the
document separator (Module 21's concatenate-and-chunk pattern).

In [ ]:
import json
import re

import tiktoken
from huggingface_hub import hf_hub_download

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # GPT-2's <|endoftext|>, id 50256

IMAGE_MD_RE = re.compile(r"!\[[^\]]*\]\([^)]*\)")

def clean_markdown(text):
    text = IMAGE_MD_RE.sub("", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


# Fetch ONLY the text file, never the full ~90GB repo (it bundles audio/image/video) -
# same important caveat identified earlier in this project's research.
genshin_wiki_path = hf_hub_download(repo_id="mrzjy/multimodal-genshin-impact", filename="genshin.jsonl", repo_type="dataset")

MAX_PAGES = 500  # small slice for this notebook\'s verification; the real run uses all 22,162

raw_pages = []
with open(genshin_wiki_path, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        text = clean_markdown(record.get("markdown") or "")
        if len(text.split()) >= 20:
            raw_pages.append(text)
        if len(raw_pages) >= MAX_PAGES:
            break

print(f"{len(raw_pages)} usable wiki pages loaded")

## 2. Budgeting the mix: total token target and Genshin share

In [ ]:
import random

# Small for this notebook\'s verification run - Module 31\'s real run sets
# TARGET_TOTAL_TOKENS to 2_500_000_000 and nothing else in this pipeline changes.
TARGET_TOTAL_TOKENS = 200_000
TARGET_GENSHIN_FRACTION = 0.10

target_genshin_tokens = int(TARGET_TOTAL_TOKENS * TARGET_GENSHIN_FRACTION)
target_general_tokens = TARGET_TOTAL_TOKENS - target_genshin_tokens
print(f"target: {target_genshin_tokens:,} Genshin tokens + {target_general_tokens:,} general tokens = {TARGET_TOTAL_TOKENS:,} total")

genshin_docs = []
genshin_total = 0
loop_count = 0
while genshin_total < target_genshin_tokens:
    loop_count += 1
    for text in raw_pages:
        ids = enc.encode_ordinary(text)
        genshin_docs.append(ids)
        genshin_total += len(ids) + 1  # +1 accounts for its EOT separator
        if genshin_total >= target_genshin_tokens:
            break

print(f"looped over the {len(raw_pages)} wiki pages {loop_count}x to reach {genshin_total:,} tokens ({len(genshin_docs)} doc-instances)")

## 3. Streaming FineWeb-Edu for the general-text share (never repeated)

In [ ]:
from datasets import load_dataset

fineweb = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

general_docs = []
general_total = 0
for row in fineweb:
    ids = enc.encode_ordinary(row["text"])
    general_docs.append(ids)
    general_total += len(ids) + 1
    if general_total >= target_general_tokens:
        break

print(f"streamed {len(general_docs)} FineWeb-Edu documents, {general_total:,} tokens")

## 4. Interleaving, concatenating, and writing the training-ready binary

Documents from both sources are shuffled together (so Genshin content is
spread throughout training, not clustered at the start of every pass over
the data), then concatenated with `EOT` separators and saved as a
`uint16` binary array — the same memmap-friendly format
[nanoGPT](https://github.com/karpathy/nanoGPT) uses, which lets Module 31
read arbitrarily large files without loading them fully into RAM (Module
21's streaming principle, applied to the final artifact too).

In [ ]:
import numpy as np

all_docs = [("genshin", d) for d in genshin_docs] + [("general", d) for d in general_docs]
random.Random(42).shuffle(all_docs)

stream = []
source_labels = []
for source, doc_ids in all_docs:
    stream.extend(doc_ids)
    stream.append(EOT)
    source_labels.extend([source] * (len(doc_ids) + 1))

tokens = np.array(stream, dtype=np.uint16)
actual_genshin_fraction = sum(1 for s in source_labels if s == "genshin") / len(source_labels)

print(f"total tokens written: {len(tokens):,}")
print(f"actual Genshin fraction: {actual_genshin_fraction:.3f} (target was {TARGET_GENSHIN_FRACTION})")
assert abs(actual_genshin_fraction - TARGET_GENSHIN_FRACTION) < 0.02

out_path = "pretrain_corpus_sample.bin"
tokens.tofile(out_path)
print(f"wrote {out_path}")

## 5. Verifying the saved file: memmap round-trip + readable content

In [ ]:
mm = np.memmap(out_path, dtype=np.uint16, mode="r")
assert len(mm) == len(tokens)
assert np.array_equal(mm[:], tokens)
print(f"memmap read back {mm.nbytes:,} bytes, matches exactly")

sample_start = len(tokens) // 3
decoded = enc.decode(mm[sample_start:sample_start + 60].tolist())
print("sample decoded chunk from the middle of the corpus:")
print(repr(decoded[:250]))

## Recap

- Built a real, verified pipeline: Genshin wiki text (repeated to hit a
  10% share) interleaved with freshly-streamed FineWeb-Edu general text,
  tokenized with the real GPT-2 tokenizer (Module 20), written as a
  memmap-friendly `uint16` binary (Module 21's streaming/chunking
  principles applied to a real artifact).
- Verified: the actual Genshin fraction landed within 2% of the 10% target,
  and the saved file round-trips exactly through `numpy.memmap`.
- **To scale this to the real run:** change exactly one constant,
  `TARGET_TOTAL_TOKENS`, from `200_000` to `2_500_000_000`, and remove the
  `MAX_PAGES` cap (use all 22,162 wiki pages). Nothing else in this
  pipeline needs to change — that's the point of building and verifying it
  properly here first.

Module 31 takes this exact corpus format and actually pretrains the ~125M
parameter model on it, on Colab Pro.